# Qwen 2.5 Coder 7B GRPO (Reinforcement Learning) Training

Trains the SFT model using Group Relative Policy Optimization (GRPO/GSPO) to improve reasoning and correctness.
Optimized for **RunPod (A40/A6000)** with **vLLM** for fast generation.

In [1]:
# These are the only extras you need for GRPO.
# The core (Torch, Unsloth, Bitsandbytes) is already perfect in the official image.
!pip install uv
!uv pip install --system trl>=0.15.0 vllm ortools wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.8/24.8 MB 67.8 MB/s  0:00:00eta 0:00:01
Using Python 3.10.12 environment at: /usr
Resolved 201 packages in 1.66s                                       
Prepared 201 packages in 50.04s                                          
░░░░░░░░░░░░░░░░░░░░ [0/201] Installing wheels...                               warning: Failed to hardlink files; falling back to full copy. This may lead to degraded performance.
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
error: Failed to install: fastsafetensors-0.3-cp310-cp310-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (fastsafetensors==0.3)
  Caused by: Failed to create directory `/usr/local/lib/python3.10/dist-packages/fastsafetensors`
  Caused by: failed to create directory `/usr/local/lib/python3.10/dist-packages/fastsafetenso

In [2]:
!pip install ortools

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.8/29.8 MB 80.5 MB/s  0:00:00m0:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [ortools]m1/2 [ortools]


In [1]:
from unsloth import FastLanguageModel
import torch
from transformers import TrainerCallback
from trl import GRPOConfig, GRPOTrainer
from datasets import load_dataset, Dataset
from ortools.sat.python import cp_model
import glob
import json
import os
import re
import sys
import wandb


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/opt/venv/lib/python3.12/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


ERROR 05-03 17:23:00 [gpt_oss_triton_kernels_moe.py:34] Failed to import Triton kernels. Please make sure your triton version is compatible. Error: No module named 'triton_kernels.routing'
🦥 Unsloth Zoo will now patch everything to make training faster!


In [2]:
from huggingface_hub import login
import wandb
import getpass
import os

print("=== Hugging Face Login ===")
hf_token = getpass.getpass("Enter your Hugging Face WRITE Token: ")
login(token=hf_token)

print("\n=== Weights & Biases Login ===")
wandb_token = getpass.getpass("Enter your W&B API Token: ")
wandb.login(key=wandb_token)

os.environ["WANDB_PROJECT"] = "Trip-Planning-GRPO" # Name of your project
os.environ["WANDB_LOG_MODEL"] = "false" # Don't upload massive model files to W&B

wandb.init(
    project="Trip-Planning-GRPO", 
    name="run-1-checkpoint-126", # Name this specific training run
    tags=["gspo", "unsloth", "ortools"]
)

print("\nAuthentication successful! Ready for training.")

=== Hugging Face Login ===


Enter your Hugging Face WRITE Token:  ········



=== Weights & Biases Login ===


Enter your W&B API Token:  ········


wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /home/unsloth/.netrc
wandb: Currently logged in as: low-h-jiun (trip-plan) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai



Authentication successful! Ready for training.


In [3]:
from unsloth import FastLanguageModel, PatchFastRL
import torch
import os
from huggingface_hub import snapshot_download

# 1. Apply the Patch BEFORE loading anything
PatchFastRL("GRPO", FastLanguageModel)

# 2. Setup paths
base_model_id = "unsloth/Qwen2.5-Coder-7b-Instruct-bnb-4bit"
checkpoint_repo = "NickolasLow1/sft-checkpoints"

# Download the adapter locally to avoid the 'subfolder' API bug
local_dir = snapshot_download(repo_id=checkpoint_repo, allow_patterns="checkpoint-126/*", token=True)
adapter_path = os.path.join(local_dir, "checkpoint-126")

# 3. Load the Base Model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = base_model_id,
    max_seq_length = 4096,
    load_in_4bit = True,
    fast_inference = True, # Critical for vLLM sync
)

# 4. CREATE LoRA structure (this is what you're missing)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",  
    random_state = 3407,                     
    use_rslora = False,
    loftq_config = None,
)

# 5. Apply your SFT Skills (LoRA)
model.load_adapter(adapter_path, adapter_name="default")
model.set_adapter("default")

# 6. THE CRITICAL STEP: Prepare for Training
# This Unsloth function automatically creates the vllm_engine AND 
# ensures the LoRA weights are synced into the generator!
FastLanguageModel.for_training(model)

print("✅ SUCCESS: Model & vLLM Engine are now perfectly synced with LoRA weights.")

Unsloth: UnslothAlignPropTrainer is already patched.
Unsloth: UnslothBCOTrainer is already patched.
Unsloth: UnslothCPOTrainer is already patched.
Unsloth: UnslothDDPOTrainer is already patched.
Unsloth: UnslothDPOTrainer is already patched.
Unsloth: UnslothGKDTrainer is already patched.
Unsloth: UnslothGRPOTrainer is already patched.
Unsloth: UnslothIterativeSFTTrainer is already patched.
Unsloth: UnslothKTOTrainer is already patched.
Unsloth: UnslothNashMDTrainer is already patched.
Unsloth: UnslothOnlineDPOTrainer is already patched.
Unsloth: UnslothORPOTrainer is already patched.
Unsloth: UnslothPPOTrainer is already patched.
Unsloth: UnslothPRMTrainer is already patched.
Unsloth: UnslothRewardTrainer is already patched.
Unsloth: UnslothRLOOTrainer is already patched.
Unsloth: UnslothSFTTrainer is already patched.
Unsloth: UnslothXPOTrainer is already patched.
INFO 05-03 17:24:02 [vllm_utils.py:724] Unsloth: Patching vLLM v1 graph capture
==((====))==  Unsloth 2026.4.8: Fast Qwen2 

/opt/venv/lib/python3.12/site-packages/pydantic/type_adapter.py:607: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `enum` - serialized value may not be as expected [field_name='mode', input_value=3, input_type=int])
  return self.serializer.to_python(


WARNING 05-03 17:24:10 [arg_utils.py:1256] The global random seed is set to 0. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.
INFO 05-03 17:24:12 [model.py:529] Resolved architecture: Qwen2ForCausalLM
INFO 05-03 17:24:12 [model.py:1549] Using max model len 4096
INFO 05-03 17:24:12 [scheduler.py:224] Chunked prefill is enabled with max_num_batched_tokens=8192.
Unsloth: vLLM Bitsandbytes config using kwargs = {'load_in_8bit': False, 'load_in_4bit': True, 'bnb_4bit_compute_dtype': 'bfloat16', 'bnb_4bit_quant_storage': 'uint8', 'bnb_4bit_quant_type': 'nf4', 'bnb_4bit_use_double_quant': True, 'llm_int8_enable_fp32_cpu_offload': False, 'llm_int8_has_fp16_weight': False, 'llm_int8_skip_modules': [], 'llm_int8_threshold': 6.0}
INFO 05-03 17:24:12 [vllm.py:689] Asynchronous scheduling is enabled.
INFO 05-03 17:24:14 [core.py:97] Initializing a V1 LLM engine (v0.16.1.dev0+g89a77b108.d20260417) with config: model='u

/opt/venv/lib/python3.12/site-packages/pydantic/type_adapter.py:607: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `enum` - serialized value may not be as expected [field_name='mode', input_value=3, input_type=int])
  return self.serializer.to_python(


INFO 05-03 17:24:15 [topk_topp_sampler.py:47] Using FlashInfer for top-p & top-k sampling.
INFO 05-03 17:24:15 [gpu_model_runner.py:4124] Starting to load model unsloth/Qwen2.5-Coder-7b-Instruct-bnb-4bit...
INFO 05-03 17:24:15 [cuda.py:367] Using FLASH_ATTN attention backend out of potential backends: ['FLASH_ATTN', 'FLASHINFER', 'TRITON_ATTN', 'FLEX_ATTENTION'].
INFO 05-03 17:24:16 [bitsandbytes_loader.py:786] Loading weights with BitsAndBytes quantization. May take a while ...
INFO 05-03 17:24:17 [weight_utils.py:579] No model.safetensors.index.json found in remote.


...

INFO 05-03 17:27:01 [core.py:278] init engine (profile, create kv cache, warmup model) took 144.93 seconds
INFO 05-03 17:27:02 [llm.py:355] Supported tasks: ('generate',)
Unsloth: Just some info: will skip parsing ['layer_norm1', 'norm1', 'norm', 'layer_norm2', 'norm2', 'ffn_norm', 'post_attention_layernorm', 'input_layernorm', 'post_layernorm', 'k_norm', 'post_feedforward_layernorm', 'pre_feedforward_layernorm', 'q_norm', 'attention_norm']


Flash Attention 2 only supports torch.float16 and torch.bfloat16 dtypes, but the current dype in Qwen2ForCausalLM is bfloat16. You should run training or inference using Automatic Mixed-Precision via the `with torch.autocast(device_type='torch_device'):` decorator, or load the model with the `dtype` argument. Example: `model = AutoModel.from_pretrained("openai/whisper-tiny", attn_implementation="flash_attention_2", dtype=torch.float16)`
Flash Attention 2 only supports torch.float16 and torch.bfloat16 dtypes, but the current dype in Qwen2Model is bfloat16. You should run training or inference using Automatic Mixed-Precision via the `with torch.autocast(device_type='torch_device'):` decorator, or load the model with the `dtype` argument. Example: `model = AutoModel.from_pretrained("openai/whisper-tiny", attn_implementation="flash_attention_2", dtype=torch.float16)`


Performing substitution for additional_keys=set()
Unsloth: Just some info: will skip parsing ['layer_norm1', 'norm1', 'norm', 'layer_norm2', 'norm2', 'ffn_norm', 'cross_attn_input_layernorm', 'post_attention_layernorm', 'input_layernorm', 'post_layernorm', 'k_norm', 'post_feedforward_layernorm', 'pre_feedforward_layernorm', 'cross_attn_post_attention_layernorm', 'q_norm', 'attention_norm']


Unsloth 2026.4.8 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


✅ SUCCESS: Model & vLLM Engine are now perfectly synced with LoRA weights.


In [5]:
# --- 1. Data Loading ---

# Load Training Data
with open("/workspace/ro4_reinforcement-learning/data/trip_planning_train.json", "r", encoding="utf-8") as f:
    train_data_raw = json.load(f)

train_data_list = []
train_source = train_data_raw.values() if isinstance(train_data_raw, dict) else train_data_raw

for item in train_source:
    p_text = item.get("prompt_0shot") or item.get("prompt")
    if not p_text:
        continue
        
    train_data_list.append({
        "prompt": [{"role": "user", "content": p_text}],
        "cities": item.get("cities"),
        "durations": item.get("durations")
    })
dataset = Dataset.from_list(train_data_list)
dataset = dataset.shuffle(seed=42)

print(f"Training Examples: {len(dataset)}")

Training Examples: 1280


In [6]:
import re
import os
import sys
import json
import tempfile
import subprocess
from ortools.sat.python import cp_model

# ==========================================
# 1. ROBUST EXTRACTION & EXECUTION HELPERS
# ==========================================

def extract_code(text):
    """Extracts python code robustly, ignoring whitespace anomalies."""
    pattern = r"```python\s+(.*?)```"
    match = re.search(pattern, text, re.DOTALL)
    if match:
        return match.group(1).strip()
    return None

def extract_json_from_output(output_str):
    """Safely extracts JSON from stdout, ignoring extra print statements."""
    try:
        start_idx = output_str.find('[')
        end_idx = output_str.rfind(']')
        start_dict = output_str.find('{')
        end_dict = output_str.rfind('}')
        
        if start_idx != -1 and end_idx != -1 and (start_dict == -1 or start_idx < start_dict):
            return json.loads(output_str[start_idx:end_idx+1])
        elif start_dict != -1 and end_dict != -1:
            return json.loads(output_str[start_dict:end_dict+1])
    except json.JSONDecodeError:
        pass
    return None

def test_code_execution(code_string):
    """Executes the code via a tempfile. Returns (is_valid, msg, parsed_json)"""
    if not code_string:
        return False, "Failed to extract code.", None
    
    with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False) as temp_file:
        temp_file.write(code_string)
        temp_file_path = temp_file.name

    try:
        result = subprocess.run([sys.executable, temp_file_path], capture_output=True, text=True, timeout=10)
        
        if result.returncode != 0:
            return False, f"Execution Error: {result.stderr.strip()[-100:]}", None
        
        output = result.stdout.strip()
        
        if "No solution" in output:
             return True, "Valid Execution", "No solution found."
             
        parsed_json = extract_json_from_output(output)
        if parsed_json is not None:
            return True, "Valid Execution", parsed_json
        else:
            return False, "Executed, but output was not valid JSON.", None
            
    except subprocess.TimeoutExpired:
        return False, "Execution Timeout (>10 seconds).", None
    finally:
        if os.path.exists(temp_file_path):
            os.remove(temp_file_path)

# ==========================================
# 2. GROUND TRUTH & NORMALIZATION 
# ==========================================

def get_ground_truth(cities_str, durations_str):
    if not cities_str or not durations_str:
        return []
    cities = cities_str.split("**")
    durations = [int(d) for d in durations_str.split("**")]
    
    gt = []
    current_start = 1
    for c, d in zip(cities, durations):
        end = current_start + d - 1
        gt.append({
            "city": c.strip().lower(), # CHANGED: lowercased to prevent string mismatch
            "start": current_start,
            "end": end
        })
        current_start = end
    return gt

def safe_int(x):
    try:
        return int(x)
    except:
        return None

def normalize(plan):
    cleaned = []

    # --- 1. Structured dict format ---
    if isinstance(plan, dict) and all(k in plan for k in ["order", "start_times", "end_times"]):
        for city in plan["order"]:
            s = safe_int(plan["start_times"].get(city))
            e = safe_int(plan["end_times"].get(city))
            if s is not None and e is not None:
                cleaned.append({
                    "city": str(city).strip().lower(),
                    "start": s,
                    "end": e
                })
        return sorted(cleaned, key=lambda x: x["start"])

    # --- 2. Unwrap nested formats ---
    if isinstance(plan, dict):
        for key in ["solutions", "trip_plan", "plan", "schedule"]:
            if key in plan:
                return normalize(plan[key])

    # --- 3. Dict-of-arrays format ---
    if isinstance(plan, dict):
        for city, dates in plan.items():
            if isinstance(dates, list) and len(dates) >= 2:
                s = safe_int(dates[0])
                e = safe_int(dates[1])
                if s is not None and e is not None:
                    cleaned.append({
                        "city": str(city).strip().lower(),
                        "start": s,
                        "end": e
                    })
        if cleaned:
            return sorted(cleaned, key=lambda x: x["start"])

    # --- 4. List of dicts format ---
    if isinstance(plan, list):
        for x in plan:
            if not isinstance(x, dict):
                continue

            city_val = x.get("city") or x.get("City") or x.get("city_name")
            if not city_val:
                continue

            s = safe_int(x.get("start") or x.get("start_day") or x.get("arrival"))
            e = safe_int(x.get("end") or x.get("end_day") or x.get("departure"))

            if s is not None and e is not None:
                cleaned.append({
                    "city": str(city_val).strip().lower(),
                    "start": s,
                    "end": e
                })

    return sorted(cleaned, key=lambda x: x["start"])

def verify_plan(generated_plan, cities_str, durations_str):
    if generated_plan is None or generated_plan == "No solution found.":
        return False

    gt_answer = normalize(get_ground_truth(cities_str, durations_str))

    try:
        norm_sol = normalize(generated_plan)
        if norm_sol == gt_answer:
            return True
    except Exception:
        pass

    return False

# ==========================================
# 3. TRL GRPO REWARD FUNCTIONS
# ==========================================

def format_reward(completions, **kwargs):
    """Reward 0.1 for correct markdown formatting (```python)"""
    rewards = []
    for code in completions:
        # Extract string from conversational format if needed
        text = code[0]['content'] if isinstance(code, list) else str(code)
        
        if "```python" in text and "```" in text.split("```python")[-1]:
            rewards.append(0.05) 
        else:
            rewards.append(0.0)
    return rewards

def syntax_reward(completions, **kwargs):
    """Reward 0.5 if the code successfully executes without crashing"""
    rewards = []
    for code in completions:
        text = code[0]['content'] if isinstance(code, list) else str(code)
        clean_code = extract_code(text)
        
        if not clean_code: 
            rewards.append(0.0)
            continue
            
        is_valid, msg, _ = test_code_execution(clean_code) 
        if is_valid or "Valid Execution" in msg:
            rewards.append(0.15) 
        else:
            rewards.append(0.0)
    return rewards

def correctness_reward(completions, cities, durations, **kwargs):
    """Reward 1.0 ONLY if the extracted JSON perfectly matches the math constraints."""
    rewards = []
    # TRL automatically passes 'cities' and 'durations' lists from your dataset kwargs!
    for code, c_str, d_str in zip(completions, cities, durations):
        text = code[0]['content'] if isinstance(code, list) else str(code)
        clean_code = extract_code(text)
        
        if not clean_code:
            rewards.append(0.0)
            continue
            
        # Run code and get JSON
        is_valid, msg, parsed_json = test_code_execution(clean_code)
        
        # Verify using SFT robust normalize logic
        if parsed_json and verify_plan(parsed_json, c_str, d_str):
            rewards.append(1.0)
        else:
            rewards.append(0.0)
            
    return rewards

In [7]:
# --- 5. Training ---
from trl import GRPOConfig, GRPOTrainer

training_args = GRPOConfig(
    output_dir = "outputs",
    run_name = "grpo_trip_planning",
    
    # === GSPO SPECIFIC ===
    importance_sampling_level = "sequence", 
    loss_type = "dr_grpo", 
    epsilon = 2e-4,           
    beta = 0.04, 
    
    # === LEARNING RATES ===
    learning_rate = 2e-6, 
    adam_beta1 = 0.9,
    adam_beta2 = 0.99,
    weight_decay = 0.1,
    warmup_ratio = 0.1,
    lr_scheduler_type = "cosine",
    
    # === PRECISION (Mandatory for RLHF) ===
    bf16 = True,  # RTX 6000 Ada supports BF16
    fp16 = False,
    
    # === BATCHING & VRAM CONTROL (48GB Safe) ===
    per_device_train_batch_size = 1, # Must be 1 to fit in 48GB
    gradient_accumulation_steps = 4,   
    num_generations = 4,               
    max_prompt_length = 512,           
    max_completion_length = 3072, 
    
    # === VLLM SETTINGS ===
    use_vllm = True, 
    vllm_gpu_memory_utilization = 0.4, # Split VRAM 50/50 between vLLM and Unsloth
    
    # === TRAINING DURATION & LOGGING ===
    max_steps = 2500, # Give it enough time; stop manually if W&B plateaus early!
    save_strategy = "steps",
    save_steps = 20, 
    save_total_limit = None, 
    logging_steps = 1,
    report_to = "wandb",
    
    gradient_checkpointing = True,
)

trainer = GRPOTrainer(
    model = model,
    ref_model = None,
    reward_funcs = [correctness_reward, syntax_reward, format_reward], 
    args = training_args,
    train_dataset = dataset,
    processing_class = tokenizer,
)

print("Starting GRPO Training ...")
trainer.train()

print("Saving Final GRPO Model...")
model.save_pretrained("grpo_final")
tokenizer.save_pretrained("grpo_final")

Starting GRPO Training ...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,280 | Num Epochs = 2 | Total steps = 2,500
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)


WARNING 05-03 17:28:00 [input_processor.py:327] vLLM has deprecated support for supporting different tokenizers for different LoRAs. By default, vLLM uses base model's tokenizer. If you are using a LoRA with its own tokenizer, consider specifying `--tokenizer [lora_path]` to use the LoRA tokenizer.
Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / correctness_reward / mean,rewards / correctness_reward / std,rewards / syntax_reward / mean,rewards / syntax_reward / std,rewards / format_reward / mean,rewards / format_reward / std
1,0.005600,0.450000,0.500000,1412.000000,1320.000000,1507.000000,0.000000,1412.000000,1320.000000,1507.000000,0.303778,0.250000,0.500000,0.150000,0.000000,0.050000,0.000000
2,0.007300,0.162500,0.075000,1463.500000,1438.000000,1481.000000,0.000000,1463.500000,1438.000000,1481.000000,0.317599,0.000000,0.000000,0.112500,0.075000,0.050000,0.000000
3,0.005700,0.162500,0.075000,1873.750000,1733.000000,1948.000000,0.000000,1873.750000,1733.000000,1948.000000,0.254276,0.000000,0.000000,0.112500,0.075000,0.050000,0.000000
4,0.005800,0.200000,0.000000,1197.750000,1142.000000,1264.000000,0.000000,1197.750000,1142.000000,1264.000000,0.369637,0.000000,0.000000,0.150000,0.000000,0.050000,0.000000
5,0.006600,0.450000,0.500000,1262.000000,1181.000000,1348.000000,0.000000,1262.000000,1181.000000,1348.000000,0.347396,0.250000,0.500000,0.150000,0.000000,0.050000,0.000000
6,0.016100,0.450000,0.500000,1091.250000,1029.000000,1164.000000,0.000000,1091.250000,1029.000000,1164.000000,0.423704,0.250000,0.500000,0.150000,0.000000,0.050000,0.000000
7,0.006300,0.200000,0.000000,1240.250000,1192.000000,1308.000000,0.000000,1240.250000,1192.000000,1308.000000,0.392276,0.000000,0.000000,0.150000,0.000000,0.050000,0.000000
8,0.005700,0.412500,0.529741,1327.500000,1317.000000,1341.000000,0.000000,1327.500000,1317.000000,1341.000000,0.356955,0.250000,0.500000,0.112500,0.075000,0.050000,0.000000
9,0.006300,0.200000,0.000000,1936.750000,1880.000000,1993.000000,0.000000,1936.750000,1880.000000,1993.000000,0.248776,0.000000,0.000000,0.150000,0.000000,0.050000,0.000000
10,0.005700,0.200000,0.000000,1222.000000,1194.000000,1288.000000,0.000000,1222.000000,1194.000000,1288.000000,0.358943,0.000000,0.000000,0.150000,0.000000,0.050000,0.000000


KeyboardInterrupt: 